# PRESCIENT Trajectory Inference on LARRY Multi-Snapshot Data

PRESCIENT (Yeo et al. 2021) is a generative neural SDE trained on cells across multiple
time points. Unlike Palantir (notebook 02), it uses day-2, day-4, and day-6 cells from
the training wells to learn a differentiation landscape, then simulates trajectories
forward from the day-2 population to predict fate probabilities.

**Pipeline:**
1. Load preprocessed LARRY data (`larry_preprocessed.h5ad` from notebook 01)
2. Split training (Wells 0+1, `fate_train`) vs held-out test replicate (Well 2, days 4/6)
3. Build PRESCIENT input from `X_pca`, time labels, and cycling-gene proliferation scores
4. Train PRESCIENT on all training time points
5. Simulate forward from every day-2 cell (Well 0) and classify endpoints
6. Evaluate against clonal ground-truth fate labels (same metrics as notebook 02)


## 1. Imports


In [ ]:
%matplotlib inline

import sys
import os
import copy
import warnings
from types import SimpleNamespace
from collections import Counter
from time import strftime, localtime

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata
import torch
import torch.nn as optim
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex

from prescient.train.model import AutoGenerator
from prescient.train.util import init as prescient_init
from prescient.train.run import run as prescient_run
import prescient.commands.train_model as prescient_train_cmd

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

PREPROCESSED_H5AD = "larry_preprocessed.h5ad"
OUTPUT_H5AD = "larry_day2_prescient.h5ad"
PRESCIENT_DATA_PT = "prescient_data/data.pt"
PRESCIENT_OUT_DIR = "prescient_models"

time_col = "Time point"
celltype_col = "Cell type annotation"
well_col = "Well"
clone_col = "clone_idx"
fate_prob_key = "prescient_fate_probabilities"

# Training hyperparameters (paper defaults; reduce TRAIN_EPOCHS for a quicker dry run)
PRETRAIN_EPOCHS = 500
TRAIN_EPOCHS = 2500
TRAIN_DT = 0.1
TRAIN_SD = 0.5
TRAIN_BATCH = 0.1
K_DIM = 500
LAYERS = 2
NUM_SIMS_PER_CELL = 10
SIM_BATCH_SIZE = 256

CYCLING_GENES = ["Mki67", "Top2a", "Pcna"]

import prescient

print(f"Python: {sys.executable}")
print(f"PyTorch: {torch.__version__}")
print(f"PRESCIENT: {prescient.__version__ if hasattr(prescient, '__version__') else 'installed'}")


## 2. Load preprocessed data


In [ ]:
adata_full = anndata.read_h5ad(PREPROCESSED_H5AD)
print(adata_full)
print("\nobs columns:", list(adata_full.obs.columns))
print("obsm keys:  ", list(adata_full.obsm.keys()))


## 3. Training vs test replicate split

Following Weinreb et al. 2020 / notebook 01:
- **Training (`fate_train`)**: Wells 0 and 1 at all time points (days 2, 4, 6)
- **Held-out test replicate**: Well 2 on days 4 and 6 (not used for PRESCIENT training)

Day-2 cells all reside in Well 0; we simulate forward from that population and
evaluate on clonal ground-truth labels stored in `fate_frac_*` columns.


In [ ]:
train_mask = adata_full.obs["fate_train"].astype(bool)
test_mask = (adata_full.obs[well_col] == 2) & (adata_full.obs[time_col] > 2)

adata_train = adata_full[train_mask].copy()
adata_test_rep = adata_full[test_mask].copy()
adata_d2 = adata_full[adata_full.obs[time_col] == 2].copy()

print(f"Training cells (Wells 0+1): {adata_train.n_obs:,}")
print(f"  Day 2: {(adata_train.obs[time_col] == 2).sum():,}")
print(f"  Day 4: {(adata_train.obs[time_col] == 4).sum():,}")
print(f"  Day 6: {(adata_train.obs[time_col] == 6).sum():,}")
print(f"Held-out Well 2 (days 4/6): {adata_test_rep.n_obs:,}")
print(f"Day-2 cells for simulation (Well 0): {adata_d2.n_obs:,}")

if clone_col in adata_d2.obs.columns:
    n_cloned = (adata_d2.obs[clone_col].fillna(-1).astype(int) >= 0).sum()
    print(f"  With clonal lineage info: {n_cloned:,} ({100 * n_cloned / adata_d2.n_obs:.1f}%)")


## 4. PRESCIENT input preparation

We pass **precomputed** `obsm['X_pca']` (50 PCs from notebook 01) directly to PRESCIENT
without recomputing PCA. Physical time points {2, 4, 6} are stored in `obs['Time point']`
and mapped to consecutive indices `0, 1, 2` for the PRESCIENT API.

Proliferation weights are derived from cycling-gene expression in the `log1p_norm` layer
(`Mki67`, `Top2a`, `Pcna`). Missing genes fall back to a uniform per-cell weight.


In [ ]:
def get_gene_expr(adata, gene_name, prefer_layer="log1p_norm"):
    """Dense expression for gene_name; prefers log1p_norm over scaled X."""
    hits = adata.var.index[adata.var.index == gene_name].tolist()
    if not hits:
        return None
    idx = adata.var.index.tolist().index(hits[0])
    if prefer_layer in adata.layers:
        col = adata.layers[prefer_layer][:, idx]
    else:
        col = adata.X[:, idx]
    return np.asarray(col.todense()).flatten() if sp.issparse(col) else np.asarray(col).flatten()


# Physical days in metadata; PRESCIENT package uses consecutive integer indices
TP_VALUES = [2.0, 4.0, 6.0]
TP_TO_IDX = {tp: i for i, tp in enumerate(TP_VALUES)}


def build_prescient_tensors(adata, tp_values=TP_VALUES):
    """Return xp list (per time index), y index list, and per-cell metadata arrays."""
    xp_list = []
    meta_tp = []
    meta_ct = []
    cell_names = []

    for tp in tp_values:
        mask = (adata.obs[time_col] == tp).values
        xp_list.append(torch.from_numpy(adata.obsm["X_pca"][mask]).float())
        meta_tp.extend([TP_TO_IDX[tp]] * int(mask.sum()))
        ct = adata.obs.loc[mask, celltype_col].astype(str).fillna("Unlabeled").values
        meta_ct.extend(ct.tolist())
        cell_names.extend(adata.obs_names[mask].tolist())

    y_idx = list(range(len(tp_values)))
    return xp_list, y_idx, np.array(meta_tp), np.array(meta_ct), cell_names


def compute_proliferation_weights(adata, xp_all, meta_tp, genes=CYCLING_GENES):
    """Per-cell proliferation score → per-timepoint weight vectors for PRESCIENT."""
    scores = np.zeros(adata.n_obs, dtype=float)
    found = []
    for gene in genes:
        expr = get_gene_expr(adata, gene)
        if expr is not None:
            mx = expr.max()
            if mx > 0:
                scores += expr / mx
            found.append(gene)

    if found:
        print(f"Cycling genes used: {found}")
    else:
        print("No cycling genes found in var — using uniform proliferation weights.")
        scores = np.ones(adata.n_obs, dtype=float)

    xp_np = xp_all.astype(np.float32)
    ay = AnnoyIndex(xp_np.shape[1], "euclidean")
    for i in range(xp_np.shape[0]):
        ay.add_item(i, xp_np[i])
    ay.build(10)

    beta = 0.1
    smoothed = scores.copy()
    prev = scores.copy()
    for _ in range(5):
        cur = np.zeros_like(prev)
        for i in range(len(prev)):
            nn = prev[ay.get_nns_by_item(i, 20)]
            cur[i] = beta * nn[0] + (1 - beta) * nn[1:].mean()
        prev = cur
        smoothed = cur

    smin, smax = smoothed.min(), smoothed.max()
    if smax > smin:
        smoothed = (smoothed - smin) / (smax - smin)
    else:
        smoothed = np.ones_like(smoothed) * 0.5

    w_list = []
    for tp_i in range(len(TP_VALUES)):
        m = meta_tp == tp_i
        w_list.append(torch.from_numpy(smoothed[m].astype(np.float32)))
    return w_list, smoothed


xp_train, y_idx, meta_tp, meta_ct, train_names = build_prescient_tensors(adata_train)
xp_all_train = adata_train.obsm["X_pca"]
w_list, growth_scores = compute_proliferation_weights(adata_train, xp_all_train, meta_tp)

print(f"PRESCIENT tensors: {len(xp_train)} time slices")
for i, tp in enumerate(TP_VALUES):
    print(f"  index {i} (day {int(tp)}): {xp_train[i].shape[0]:,} cells, w mean={w_list[i].mean():.3f}")

os.makedirs(os.path.dirname(PRESCIENT_DATA_PT) or ".", exist_ok=True)
torch.save({
    "xp": xp_train,
    "y": y_idx,
    "y_days": TP_VALUES,
    "w": w_list,
    "celltype": meta_ct,
    "tps": meta_tp,
    "names": train_names,
}, PRESCIENT_DATA_PT)
print(f"Saved PRESCIENT data to {PRESCIENT_DATA_PT}")


## 5. Train PRESCIENT

Train on all `fate_train` cells across time indices 0 → 1 → 2 (days 2 → 4 → 6).


In [ ]:
def prescient_train_init(args):
    """Build training config from data.pt (mirrors prescient.commands.train_model.train_init)."""
    data_pt = torch.load(args.data_path, weights_only=False)
    x = data_pt["xp"]
    y = data_pt["y"]
    weight = data_pt["w"]

    a = copy.copy(args)
    if args.weight_name is not None:
        a.weight = args.weight_name

    name = (
        "{weight}-{activation}_{layers}_{k_dim}-{train_tau}"
    ).format(**a.__dict__)

    a.out_dir = os.path.join(args.out_dir, name, f"seed_{a.seed}")
    config = prescient_train_cmd.init_config(a)

    config.x_dim = x[0].shape[-1]
    config.t = y[-1] - y[0]
    config.start_t = y[0]
    config.train_t = y[1:]

    y_start = y[config.start_t]
    y_ = [yy for yy in y if yy > y_start]
    w_ = weight[config.start_t]
    w = {(y_start, yy): torch.from_numpy(np.exp((yy - y_start) * w_.numpy())) for yy in y_}

    return x, y, w, config


class TrainArgs(SimpleNamespace):
    pass


train_args = TrainArgs(
    seed=SEED,
    gpu=0,
    no_cuda=not torch.cuda.is_available(),
    data_path=PRESCIENT_DATA_PT,
    out_dir=PRESCIENT_OUT_DIR,
    weight_name="cycling",
    loss="euclidean",
    k_dim=K_DIM,
    activation="softplus",
    layers=LAYERS,
    pretrain_epochs=PRETRAIN_EPOCHS,
    train_epochs=TRAIN_EPOCHS,
    train_lr=0.01,
    train_dt=TRAIN_DT,
    train_sd=TRAIN_SD,
    train_tau=1e-6,
    train_batch=TRAIN_BATCH,
    train_clip=0.25,
    save=500,
    pretrain=True,
    train=True,
)

os.makedirs(PRESCIENT_OUT_DIR, exist_ok=True)
print(f"Training PRESCIENT ({TRAIN_EPOCHS} epochs, seed={SEED}) ...")
prescient_run(train_args, prescient_train_init)
print("Training complete.")


## 6. Load trained model and reference classifier

We classify simulated endpoints by k-nearest neighbors in PCA space against **training**
cells with a cell-type annotation (Wells 0+1, all time points).


In [ ]:
# Locate trained model directory
model_name = (
    f"{train_args.weight_name}-{train_args.activation}_{train_args.layers}_{train_args.k_dim}-{train_args.train_tau}"
)
model_dir = os.path.join(PRESCIENT_OUT_DIR, model_name, f"seed_{SEED}")
config_pt = os.path.join(model_dir, "config.pt")
config = SimpleNamespace(**torch.load(config_pt, weights_only=False))

device, _ = prescient_init(train_args)
net = AutoGenerator(config)
net.to(device)

best_pt = os.path.join(model_dir, "train.epoch_best.pt")
if not os.path.exists(best_pt):
    # fall back to last checkpoint
    ckpts = sorted([f for f in os.listdir(model_dir) if f.startswith("train.epoch_") and f.endswith(".pt")])
    best_pt = os.path.join(model_dir, ckpts[-1]) if ckpts else None
print(f"Loading weights from {best_pt}")
checkpoint = torch.load(best_pt, map_location=device, weights_only=False)
net.load_state_dict(checkpoint["model_state_dict"])
net.eval()

# kNN reference: annotated training cells
ref_mask = adata_train.obs[celltype_col].notna().values
ref_pca = adata_train.obsm["X_pca"][ref_mask]
ref_labels = adata_train.obs.loc[ref_mask, celltype_col].astype(str).values
print(f"Reference cells for classification: {ref_pca.shape[0]:,}")

knn_clf = NearestNeighbors(n_neighbors=10, metric="euclidean")
knn_clf.fit(ref_pca)

# Simulation length: physical days 2 → 6 in steps of train_dt
num_steps = int(np.round((TP_VALUES[-1] - TP_VALUES[0]) / config.train_dt))
print(f"Forward simulation steps (day 2 → 6): {num_steps}")


## 7. Simulate forward from day-2 cells and compute fate probabilities

For each day-2 cell we run `NUM_SIMS_PER_CELL` stochastic forward simulations,
classify each endpoint with the kNN reference, and record the fraction of simulations
assigned to each lineage. Column names match the `fate_frac_*` ground-truth labels.


In [ ]:
fate_frac_cols = sorted(c for c in adata_d2.obs.columns if c.startswith("fate_frac_"))
# Column names match fate_frac_* after stripping prefix (same index/columns as notebook 02 eval)
branch_names = [c.replace("fate_frac_", "") for c in fate_frac_cols]
# Map fate_frac column stems to Cell type annotation strings for kNN classification
LINEAGE_TO_CT = {
    "Baso": "Baso",
    "Ccr7_DC": "Ccr7 DC",
    "Eos": "Eos",
    "Erythroid": "Erythroid",
    "Lymphoid": "Lymphoid",
    "Mast": "Mast",
    "Meg": "Meg",
    "Monocyte": "Monocyte",
    "Neutrophil": "Neutrophil",
    "Undifferentiated": "Undifferentiated",
    "pDC": "pDC",
}
ct_to_lineage = {v: k for k, v in LINEAGE_TO_CT.items()}


def classify_endpoints(xp_end, knn, labels):
    _, idx = knn.kneighbors(xp_end)
    preds = []
    for neighbors in idx:
        votes = Counter(labels[neighbors])
        preds.append(votes.most_common(1)[0][0])
    return preds


def simulate_batch(xp_start, model, config, num_steps, device):
    xp_i = torch.from_numpy(xp_start).float().to(device)
    with torch.no_grad():
        for _ in range(num_steps):
            z = torch.randn(xp_i.shape[0], xp_i.shape[1], device=device) * config.train_sd
            xp_i = model._step(xp_i, dt=config.train_dt, z=z)
    return xp_i.detach().cpu().numpy()


xp_d2 = adata_d2.obsm["X_pca"]
n_d2 = xp_d2.shape[0]
fate_counts = np.zeros((n_d2, len(branch_names)), dtype=float)

print(f"Simulating {n_d2:,} day-2 cells × {NUM_SIMS_PER_CELL} trajectories each ...")

for start in range(0, n_d2, SIM_BATCH_SIZE):
    end = min(start + SIM_BATCH_SIZE, n_d2)
    xp_batch = xp_d2[start:end]

    for _ in range(NUM_SIMS_PER_CELL):
        xp_end = simulate_batch(xp_batch, net, config, num_steps, device)
        preds = classify_endpoints(xp_end, knn_clf, ref_labels)
        for j, ct in enumerate(preds):
            lin = ct_to_lineage.get(ct)
            if lin is None:
                # try direct match on sanitized names
                lin = ct.replace(" ", "_") if ct.replace(" ", "_") in LINEAGE_TO_CT else None
            if lin is not None and lin in branch_names:
                fate_counts[start + j, branch_names.index(lin)] += 1

    if (end % (SIM_BATCH_SIZE * 10) == 0) or end == n_d2:
        print(f"  {end:,} / {n_d2:,} cells")

fate_probs = fate_counts / NUM_SIMS_PER_CELL
fate_probs_df = pd.DataFrame(
    fate_probs,
    index=adata_d2.obs_names,
    columns=branch_names,
)
# Renormalize rows (unclassified sims may leave mass < 1)
row_sums = fate_probs_df.sum(axis=1).replace(0, np.nan)
fate_probs_df = fate_probs_df.div(row_sums, axis=0).fillna(0)

print(f"Fate probability matrix: {fate_probs_df.shape}")
print(fate_probs_df.sum(axis=1).describe())

adata_d2.obsm[fate_prob_key] = fate_probs_df.values
adata_d2.uns["predicted_branch_names"] = branch_names


## 7b. Visualize fate probabilities on UMAP


In [ ]:
if "X_umap" in adata_d2.obsm:
    umap_d2 = adata_d2.obsm["X_umap"]
else:
    sc.pp.neighbors(adata_d2, n_neighbors=30, n_pcs=50, use_rep="X_pca")
    sc.tl.umap(adata_d2, random_state=SEED)
    umap_d2 = adata_d2.obsm["X_umap"]

# Show a few major lineages
show_lineages = [l for l in ["Neutrophil", "Monocyte", "Erythroid", "Lymphoid", "Meg", "Baso"] if l in branch_names]
ncols = min(len(show_lineages), 3)
nrows = (len(show_lineages) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).flatten()

for i, lin in enumerate(show_lineages):
    j = branch_names.index(lin)
    sc_ = axes[i].scatter(umap_d2[:, 0], umap_d2[:, 1], s=3, alpha=0.7,
                          c=fate_probs_df[lin].values, cmap="RdBu_r", vmin=0, vmax=1)
    plt.colorbar(sc_, ax=axes[i])
    axes[i].set_title(f"P(→ {lin})")
    axes[i].set_xlabel("UMAP 1")
    axes[i].set_ylabel("UMAP 2")

for ax in axes[len(show_lineages):]:
    ax.set_visible(False)

plt.suptitle("PRESCIENT fate probabilities", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 8. Evaluate against clonal ground-truth fate labels

For day-2 cells with clonal fate fractions we compare PRESCIENT's predicted lineage
probabilities to observed `fate_frac_*` values using the same metrics as notebook 02.


In [ ]:
# Fate fractions were computed in notebook 01 and saved as fate_frac_* columns
fate_frac_cols = sorted(c for c in adata_d2.obs.columns if c.startswith("fate_frac_"))
print(f"Fate fraction columns found: {len(fate_frac_cols)}")
for c in fate_frac_cols:
    print(f"  {c}")

has_fate = pd.Series(False, index=adata_d2.obs_names)
if fate_frac_cols:
    has_fate = adata_d2.obs[fate_frac_cols[0]].notna()
    n = has_fate.sum()
    print(f"\nDay-2 cells with ground-truth fate fractions: {n:,} / {adata_d2.n_obs:,}")
    if n > 0:
        print("\nMean fate fractions across cloned day-2 cells:")
        means = adata_d2.obs.loc[has_fate, fate_frac_cols].mean().sort_values(ascending=False)
        print(means.rename(lambda c: c.replace("fate_frac_", "")).round(3).to_string())
else:
    print("No fate_frac_* columns found in adata_d2.obs.")


In [ ]:
fate_df_norm = None

if fate_frac_cols and has_fate.sum() > 0:
    fate_df_norm = adata_d2.obs.loc[has_fate, fate_frac_cols].copy()
    fate_df_norm = fate_df_norm.loc[:, fate_df_norm.sum() > 0]
    fate_df_norm.columns = [c.replace("fate_frac_", "") for c in fate_df_norm.columns]
    print(f"Ground-truth fate matrix: {fate_df_norm.shape[0]:,} cells × {fate_df_norm.shape[1]} lineages")
    print(f"Lineages: {list(fate_df_norm.columns)}")
    print(fate_df_norm.head(3).round(3))
else:
    print("Skipping: no fate fractions available (see cell above).")


In [ ]:
corr_df = None

if fate_df_norm is not None and fate_prob_key in adata_d2.obsm:
    branch_names = adata_d2.uns.get(
        "predicted_branch_names",
        [f"Branch_{i}" for i in range(adata_d2.obsm[fate_prob_key].shape[1])]
    )
    bp_df = pd.DataFrame(
        adata_d2.obsm[fate_prob_key],
        index=adata_d2.obs_names,
        columns=branch_names,
    )

    common = fate_df_norm.index.intersection(bp_df.index)
    print(f"Cells with both predicted fate probs and fate labels: {len(common):,}")
    fate_sub = fate_df_norm.loc[common]
    bp_sub   = bp_df.loc[common]

    corr_mat = np.zeros((len(fate_sub.columns), len(bp_sub.columns)))
    for i, fc in enumerate(fate_sub.columns):
        for j, bc in enumerate(bp_sub.columns):
            corr_mat[i, j], _ = stats.pearsonr(fate_sub[fc].values, bp_sub[bc].values)

    corr_df = pd.DataFrame(corr_mat, index=fate_sub.columns, columns=bp_sub.columns)
    print("\nPearson r (observed fate fraction × predicted branch probability):")
    print(corr_df.round(3).to_string())

    # Reorder rows: identified lineages aligned to diagonal, unidentified appended after
    identified = [c for c in corr_df.columns if c in corr_df.index]
    unidentified = [r for r in corr_df.index if r not in corr_df.columns]
    corr_df_plot = corr_df.loc[identified + unidentified]

    fig_w = max(6, len(bp_sub.columns))
    fig_h = max(4, len(fate_sub.columns))
    fig_h = max(4, len(corr_df_plot))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(corr_df_plot, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title("Predicted branch probs vs observed fate fractions (Pearson r)")
    ax.set_xlabel("Predicted branch")
    ax.set_ylabel("Observed fate (clonal)")
    plt.tight_layout()
    plt.savefig("prescient_corr_heatmap.png", dpi=80, bbox_inches="tight")
    plt.show()
else:
    print("Skipping correlation heatmap: no fate fractions or branch probs available.")


>##### Observe diagonal for the first 6 rows/columns; Weinreb 2020 reports r ≈ 0.3–0.4 for methods that work.


In [ ]:
# Weinreb-style Neutrophil–Monocyte bias score
# Both column names use underscores from the fate_frac_* convention
NEU_KEYS = ["Neutrophil", "Neu"]
MONO_KEYS = ["Monocyte",   "Mono"]

if fate_df_norm is not None and corr_df is not None:
    neu_col = next((c for c in fate_df_norm.columns if c in NEU_KEYS),  None)
    mono_col = next((c for c in fate_df_norm.columns if c in MONO_KEYS), None)

    if neu_col and mono_col:
        denom = fate_sub[neu_col] + fate_sub[mono_col]
        obs_bias = (fate_sub[neu_col] / denom.replace(0, np.nan)).dropna()

        best_branch = corr_df.loc[neu_col].abs().idxmax()
        pred_bias = bp_sub.loc[obs_bias.index, best_branch].values
        r, p = stats.pearsonr(pred_bias, obs_bias.values)

        print(f"Neu–Mono bias  Pearson r: {r:.4f}  (p={p:.2e})")
        print(f"(Predicted branch '{best_branch}' used as Neutrophil predictor)")

        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(pred_bias, obs_bias.values, s=10, alpha=0.5, color="steelblue")
        ax.set_xlabel(f"Predicted P({best_branch})")
        ax.set_ylabel("Observed Neu / (Neu + Mono)")
        ax.set_title(f"Neu–Mono fate bias   r = {r:.3f}")
        plt.tight_layout()
        plt.savefig("prescient_neu_mono_bias.png", dpi=80, bbox_inches="tight")
        plt.show()
    else:
        print(f"Neu/Mono columns not in fate_df_norm. Available: {list(fate_df_norm.columns)}")
else:
    print("Skipping: correlation matrix not available.")


>##### Interpretation: among day-2 cells with clone assignments, many clones produce both Neutrophils and Monocytes at day 4/6. Some clones produce mostly Neutrophils, some mostly Monocytes. Can PRESCIENT detect it from gene expression alone? <br> Y axis - Neu / (Neu + Mono); for each cloned day-2 cell, what fraction of its clone's Neu+Mono descendants ended up as Neutrophils? 0 = pure Monocyte fate, 1 = pure Neutrophil fate. <br> X axis - Predicted P(Neutrophil branch) for that same cell (branch with highest |r| to observed Neutrophil in the Pearson matrix). <br> We expect points trending upward - cells assigned high predicted Neutrophil probability should also have high observed Neu fraction.


## 8c. Wasserstein-1 distance (per-cell fate distribution comparison)

Earth Mover's Distance between the predicted fate distribution and the observed clonal fate distribution, restricted to lineages present in both. Lower = better. Lineages are treated as points on a 1-D axis (alphabetical order); cost of moving mass between adjacent lineages is 1.


In [ ]:
from scipy.stats import wasserstein_distance as _w1

if fate_df_norm is not None and fate_prob_key in adata_d2.obsm:
    branch_names_wd = adata_d2.uns.get(
        "predicted_branch_names",
        [f"Branch_{i}" for i in range(adata_d2.obsm[fate_prob_key].shape[1])]
    )
    bp_df_wd = pd.DataFrame(
        adata_d2.obsm[fate_prob_key],
        index=adata_d2.obs_names,
        columns=branch_names_wd,
    )

    shared = sorted(set(fate_df_norm.columns) & set(bp_df_wd.columns))
    print(f"Shared lineages (predicted branches ∩ observed fate labels): {shared}")
    print(f"  ({len(shared)} of {len(fate_df_norm.columns)} observed lineages covered by predicted branches)")

    common_wd = fate_df_norm.index.intersection(bp_df_wd.index)
    fate_sub_wd = fate_df_norm.loc[common_wd, shared].values.astype(float)
    bp_sub_wd = bp_df_wd.loc[common_wd, shared].values.astype(float)
    lineage_vals = np.arange(len(shared), dtype=float)

    per_cell_wd = []
    for obs_vec, pred_vec in zip(fate_sub_wd, bp_sub_wd):
        obs_s, pred_s = obs_vec.sum(), pred_vec.sum()
        if obs_s > 0 and pred_s > 0:
            per_cell_wd.append(
                _w1(lineage_vals, lineage_vals,
                    obs_vec / obs_s, pred_vec / pred_s)
            )

    per_cell_wd = np.array(per_cell_wd)
    print(f"\nWasserstein-1 distance ({len(shared)} shared lineages, {len(per_cell_wd):,} cells):")
    print(f"  Mean:    {per_cell_wd.mean():.4f}")
    print(f"  Median:  {np.median(per_cell_wd):.4f}")
    print(f"  Std:     {per_cell_wd.std():.4f}")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(per_cell_wd, bins=50, color="steelblue", edgecolor="none")
    ax.axvline(per_cell_wd.mean(), color="red", linestyle="--",
               label=f"Mean = {per_cell_wd.mean():.3f}")
    ax.set_xlabel("Wasserstein-1 distance (predicted vs observed fate)")
    ax.set_ylabel("Cells")
    ax.set_title("Per-cell Wasserstein distance: predicted vs clonal ground truth")
    ax.legend()
    plt.tight_layout()
    plt.savefig("prescient_wasserstein.png", dpi=80, bbox_inches="tight")
    plt.show()
else:
    print("Skipping: fate fractions or PRESCIENT probabilities not available.")


## 9. Save


In [ ]:
print(f"Saving to {OUTPUT_H5AD} ...")
adata_d2.write_h5ad(OUTPUT_H5AD)

fate_probs_df.to_csv("prescient_day2_fate_probs.csv")
print("CSV saved to prescient_day2_fate_probs.csv")

result_cols = [c for c in adata_d2.obs.columns
               if c.startswith("fate_frac_") or c in [time_col, clone_col, celltype_col]]
adata_d2.obs[result_cols].join(fate_probs_df).to_csv("prescient_day2_results.csv")
print("CSV saved to prescient_day2_results.csv")


In [ ]:
print("=" * 55)
print("PRESCIENT ANALYSIS COMPLETE")
print("=" * 55)
print(f"  Training cells: {adata_train.n_obs:,}")
print(f"  Day-2 cells:    {adata_d2.n_obs:,}")
print(f"  Model dir:      {model_dir}")
print(f"  Branch names:   {adata_d2.uns.get('predicted_branch_names', 'N/A')}")
print(f"  Output h5ad:    {OUTPUT_H5AD}")


---
## Eval-only checkpoint

Run the cell below on a **fresh kernel** to restore all variables needed by the section 8
eval cells without re-running training or simulation. Requires `larry_day2_prescient.h5ad`
and `prescient_day2_fate_probs.csv`.


In [ ]:
%matplotlib inline

import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import anndata

warnings.filterwarnings("ignore")

OUTPUT_H5AD = "larry_day2_prescient.h5ad"
fate_prob_key = "prescient_fate_probabilities"

print(f"Loading {OUTPUT_H5AD} ...")
adata_d2 = anndata.read_h5ad(OUTPUT_H5AD)
fate_frac_cols = sorted(c for c in adata_d2.obs.columns if c.startswith("fate_frac_"))
has_fate = (
    adata_d2.obs[fate_frac_cols[0]].notna()
    if fate_frac_cols
    else pd.Series(False, index=adata_d2.obs_names)
)

fate_df_norm = None
if fate_frac_cols and has_fate.sum() > 0:
    fate_df_norm = adata_d2.obs.loc[has_fate, fate_frac_cols].copy()
    fate_df_norm = fate_df_norm.loc[:, fate_df_norm.sum() > 0]
    fate_df_norm.columns = [c.replace("fate_frac_", "") for c in fate_df_norm.columns]

branch_names = (
    adata_d2.uns.get(
        "predicted_branch_names",
        [f"Branch_{i}" for i in range(adata_d2.obsm[fate_prob_key].shape[1])]
    )
    if fate_prob_key in adata_d2.obsm else []
)
bp_df = (
    pd.DataFrame(adata_d2.obsm[fate_prob_key], index=adata_d2.obs_names, columns=branch_names)
    if fate_prob_key in adata_d2.obsm else None
)
fate_probs_df = bp_df

corr_df = fate_sub = bp_sub = None
if fate_df_norm is not None and bp_df is not None:
    common = fate_df_norm.index.intersection(bp_df.index)
    fate_sub = fate_df_norm.loc[common]
    bp_sub = bp_df.loc[common]

    corr_mat = np.zeros((len(fate_sub.columns), len(bp_sub.columns)))
    for i, fc in enumerate(fate_sub.columns):
        for j, bc in enumerate(bp_sub.columns):
            corr_mat[i, j], _ = stats.pearsonr(fate_sub[fc].values, bp_sub[bc].values)
    corr_df = pd.DataFrame(corr_mat, index=fate_sub.columns, columns=bp_sub.columns)

print("\nAll eval-metric variables restored:")
print(f"  adata_d2     {adata_d2.n_obs:,} cells, obsm keys: {list(adata_d2.obsm.keys())}")
print(f"  fate_df_norm {fate_df_norm.shape if fate_df_norm is not None else None}")
print(f"  fate_sub     {fate_sub.shape if fate_sub is not None else None}")
print(f"  bp_sub       {bp_sub.shape if bp_sub is not None else None}")
print(f"  corr_df      {corr_df.shape if corr_df is not None else None}")
print(f"  branch_names {branch_names}")
print("\nReady — run any eval cell in section 8.")
